# Code used for re-drawing samples

The random sample dataset is composed of marginal ice zone (MIZ) locations and dates from 9 Arctic regions. The dates are selected based on the climatology of sea ice cover for each region. Each scene is 100 km by 100 km and centered on a location where the NSIDC climate data record v5 (CDR) of sea ice concentration shows between 15 and 85%. The CDR is used to find candidate samples, as the dataset can be accessed quickly. Since the resolution is 25 km, each candidate scene only contains a few CDR pixels. We refine the candidate samples by computing the sea ice cover from the MASIE 4 km data. Only scenes with at least 25% sea ice fraction (area with SIC > 15%) are considered. MODIS images are available since 2000 (terra) and 2002 (aqua), however the MASIE sea ice extent imagery is only available since 2006. So we draw our sample from 2006 to 2023 (the last full year of NSIDC data at time of writing).

Sample selection needs to balance two competing issues:
(a) the size of the image needs to be large enough that there is a useful range of floe sizes that can be resolved
(b) the number of scenes should be large enough that we can consider it a representative random sample
Because the time needed to manually annotate an image is proportional to the image area, the number of images that can be analyzed is strongly limited by the image size. 

Initially, nine regions spanning the circumpolar Arctic seasonal ice zones were selected The regions were selected to have equal area. The regions were subsquently redefined to avoid overlaps and to align better with standard marginal see boundaries.

We perform stratified random sampling to select a set of images for the validation dataset.
We use the NSIDC Sea Ice Concentration Climate Data Record (SIC CDR) to identify periods of time that each region contains some overlap with the sea ice region. We acknowledge that uncertainty in SIC CDR is high during the summer. Since the purpose of using the SIC CDR in the random sampling is to find a wide range of sea ice conditions, rather than to make inferences about sea ice physics in these regions, we consider the SIC CDR suitable for this purpose.

Within each region, we follow the following procedure
* Randomly choose a start year and a date between March and region-specific end date (max September 30)
* Select a center point (x, y) from the CDR grid that is at least 50 km from the region border
* Calculate sea ice fraction for the 100 km box centered at the point
* If the sea ice fraction is at least 0.25, add date and center point to list of cases
* Check the number of cases.
   * If the minimum number of cases per month is met, move to the next region.
    
We start by generating a set of samples larger than is needed for the validation dataset, saving 10 candidate cases per month, per region. We then stratify the sample to get minimum 3 samples per month, with randomly chosen additional samples chosen to bring the number to 21 dates per region for locations where sea ice is present less than 7 months of the summer. Thus, there are more samples for March in the Sea of Okhostk than in the Beaufort Sea.

Developing the random sample dataset began in Spring 2024. Initial samples were drawn using the same functions as below, but for equal-area regions. Because the region boundaries overlapped, some points had ambiguous region membership. We also found cases where the sea ice concentration data from the NSIDC was biased high, due to issues with proximity to the coast. 

We include some samples drawn from the same sampling procedure with slightly different region definitions. We refined the region definitions to avoid overlaps and to align better with the IHO marginal sea definitions, and we extended the time range an additional year. To ensure that the sampling was still random according to the procedure described above, we drew a large set of new sample locations, then computed the relative fraction within the old domain and with the modified domain. Using the relative fractions from the overlapping and non-overlapping regions, we drew samples from the new regions until the proportion was correct.

The code below takes the original sample, used in the Simon Hatcher's batchelor's thesis (Hatcher 2024), then resamples the region to create a random sample representative of the new region. Consider a region $X$ and a new, overlapping region $Y$. We posses a sample of $n_X$ points from region $X$ and wish to form a sample of points from $Y$. Drawing a random sample of $m$ points from $Y$, we separate them into $m_{X}$ points from $Y \cap X$ (i.e., the points in $Y$ that are also in $X$), and $m_{Y \setminus X}$ points that are in region of $Y$ excluding $X$. Thus, $m_X + m_{Y \setminus X} = n_{Y} = n_{X}$. To extend our original sample, we randomly select up to $m_X$ points from the $n_X$ original points. If $n_X < m_X$, we replace $n_X$ points from the $m_X$ new points with points from the old sample, and keep $m_X - n_X$ points from the new sample. If $n_X >= m_X$, we draw $m_X$ of the originally sampled points at random to keep in the new sample.



### Disclaimer
This notebook is for archival purposes to show code used in interactive sessions, it is not intended to be run end-to-end.

In [ ]:
import os
import pyproj
import numpy as np
import pandas as pd
import xarray as xr

# Load sea ice fraction data
sif_data = pd.read_csv('../data/metadata/daily_sea_ice_fraction.csv', index_col=0, parse_dates=True)
threshold = 0.05
end_doy = sif_data.groupby(sif_data.index.dayofyear).quantile(0.90).apply(lambda x: x[x >= threshold].index.max())
start_doy = pd.Series(pd.to_datetime('2003-03-01').dayofyear, index=end_doy.index)

# Load region definitions and add start and end dates
regions = pd.read_csv('../data/metadata/region_definitions.csv', index_col=0)
regions.loc[end_doy.index, 'end_doy'] = end_doy.values
regions.loc[start_doy.index, 'start_doy'] = start_doy.values
regions.loc[start_doy.index, 'start_mmdd'] = (pd.to_timedelta(regions['start_doy'] - 1, unit='D') + pd.to_datetime('2001-01-01 00:00')).dt.strftime('%b %d')
regions.loc[end_doy.index, 'end_mmdd'] = (pd.to_timedelta(regions['end_doy'] - 2, unit='D') + pd.to_datetime('2001-01-01 00:00')).dt.strftime('%b %d')
regions.sort_values('center_lon').loc[:, ['start_doy', 'end_doy', 'start_mmdd', 'end_mmdd']].to_csv('../data/metadata/sample_start_end_times.csv')

# Updating the sample
1. Flag the cases in the original sample that are within the new time range and location range.
2. Draw a new sample using the original process, using the new bounds.
3. Determine the number of points $n$ in the new sample that fall within the old bounds.
    1. If $n$ is less than or equal to the number of points in the original sample, keep $n$ points from the original. 
    2. Keep the points in the new sample that fall in the new boundaries, outside the old boundaries.

In [ ]:
def in_box(x, y, left_x, right_x, lower_y, upper_y):
    """Convenience function that checks whether point x, y is in region.
    Returns True if x, y in the box."""
    margin = 50e3 # margin in meters
    if left_x <= x - margin:
        if right_x >= x + margin:
            if lower_y <= y - margin:
                if upper_y >= y + margin:
                    return True
    return False
    
def in_new_minus_old(point, region, new_regions, old_regions):
    """Checks whether point is in the set defined by the new region
    (regions.loc[region,:]) excluding the area of the old region
    (old_regions.loc[region,:])"""
        
    old_xmin = old_regions.loc[region, 'left_x']
    old_xmax = old_regions.loc[region, 'right_x']
    old_ymin = old_regions.loc[region, 'lower_y']
    old_ymax = old_regions.loc[region, 'upper_y']

    new_xmin = new_regions.loc[region, 'left_x']
    new_xmax = new_regions.loc[region, 'right_x']
    new_ymin = new_regions.loc[region, 'lower_y']
    new_ymax = new_regions.loc[region, 'upper_y']
    
    
    in_old = in_box(point.center_x, point.center_y, old_xmin, old_xmax, old_ymin, old_ymax)
    in_new = in_box(point.center_x, point.center_y, new_xmin, new_xmax, new_ymin, new_ymax)
    return in_new & ~in_old

In [ ]:
regions_to_sample = ['bering_chukchi_seas', 'beaufort_sea', 'hudson_bay', 'barents_kara_seas', 'greenland_sea']
results = {}
for region in regions_to_sample:
    row = regions.loc[region]
    test = find_candidates(n_samples=2000,
                    region_info=row,
                    dx=100,
                    sic_path=sic_dataloc,
                    sif_threshold=0.25,
                    margin=50, start_year=2003, end_year=2022,
                    min_samples_per_month=200) # Margin keeps the edges of boxes inside the outer bounds.
    
    test['in_new_minus_old'] = False
    test.loc[[name for name, row in test.iterrows() if in_new_minus_old(row, region, regions, old_regions)], 'in_new_minus_old'] = True
    results[region] = test
    print(region)

In [ ]:
pplt.rc['reso'] = 'med'
pplt.rc['cartopy.circular'] = False

crs = ccrs.CRS('epsg:3413')
fig, ax = pplt.subplots(width=4.5, proj='npstere', proj_kw={'lon_0': -45})
ax.format(land=True, color='k', boundinglat=52, landzorder=0, latmax=90)
ax.set_extent([-3.5e6, 2.8e6, -4e6, 4.4e6], crs=crs)

for region in results:
    xbox = np.array([old_regions.loc[region, coord] for coord in ['left_x', 'left_x', 'right_x', 'right_x', 'left_x']])
    ybox = np.array([old_regions.loc[region, coord] for coord in ['lower_y', 'upper_y', 'upper_y', 'lower_y', 'lower_y']])
    ax.plot(xbox, ybox, transform=crs, color='gray')
    
    
    xbox = np.array([regions.loc[region, coord] for coord in ['left_x', 'left_x', 'right_x', 'right_x', 'left_x']])
    ybox = np.array([regions.loc[region, coord] for coord in ['lower_y', 'upper_y', 'upper_y', 'lower_y', 'lower_y']])
    ax.plot(xbox, ybox, transform=crs)
    
    ax.scatter(results[region]['center_lon'], results[region]['center_lat'], color='lightgray', marker='.', ms=2)

In [ ]:
init_case_list = pd.read_csv('../../ift_cal-val/data/metadata/validation_dataset_case_list.csv', parse_dates =['start_date'], index_col='case_number')
# Rename regions where appropriate
for row, data in init_case_list.iterrows():
    if data.region == 'bering_strait':
        init_case_list.loc[row, 'region'] = 'bering_chukchi_seas'
    elif data.region == 'chukchi_east_siberian_seas':
        init_case_list.loc[row, 'region'] = 'east_siberian_sea'
        
init_case_list['in_new_region'] = False
for name, row in init_case_list.iterrows():
    init_case_list.loc[name, 'in_new_region'] = in_box(row.center_x, row.center_y, regions.loc[row.region, 'left_x'],
                                           regions.loc[row.region, 'right_x'], regions.loc[row.region, 'lower_y'],
                                           regions.loc[row.region, 'upper_y'])
init_case_list['in_time_range'] = init_case_list.start_date >= pd.to_datetime('2006-01-01')

## Drawing new samples
This section draws new samples to match the improved boundaries.

Along with the change in boundaries to better match standard marginal sea definitions and avoid overlaps, we focus on the time range where MASIE sea ice extent data is available. Thus we need to see how many new samples to draw to replace the discarded samples. 

In [ ]:
# Show the original sample and the changed bounds
# Blue for inside, red for outside the boundaries
pplt.rc['reso'] = 'med'
pplt.rc['cartopy.circular'] = False

crs = ccrs.CRS('epsg:3413')
fig, ax = pplt.subplots(width=4.5, proj='npstere', proj_kw={'lon_0': -45})
ax.format(land=True, color='k', boundinglat=52, landzorder=0, latmax=90)
ax.set_extent([-3.5e6, 2.8e6, -4e6, 4.4e6], crs=crs)

for region in regions.index:
    xbox = np.array([old_regions.loc[region, coord] for coord in ['left_x', 'left_x', 'right_x', 'right_x', 'left_x']])
    ybox = np.array([old_regions.loc[region, coord] for coord in ['lower_y', 'upper_y', 'upper_y', 'lower_y', 'lower_y']])
    ax.plot(xbox, ybox, transform=crs, color='gray')
    
    xbox = np.array([regions.loc[region, coord] for coord in ['left_x', 'left_x', 'right_x', 'right_x', 'left_x']])
    ybox = np.array([regions.loc[region, coord] for coord in ['lower_y', 'upper_y', 'upper_y', 'lower_y', 'lower_y']])
    ax.plot(xbox, ybox, transform=crs)
    

good_points = init_case_list.in_new_region & init_case_list.in_time_range

ax.scatter(init_case_list.loc[good_points, 'center_lon'],
           init_case_list.loc[good_points, 'center_lat'],
           color='steelblue', edgecolor='k', marker='o', ms=15)
ax.scatter(init_case_list.loc[~init_case_list.in_time_range, 'center_lon'],
           init_case_list.loc[~init_case_list.in_time_range, 'center_lat'], color='gold', edgecolor='k', marker='o', ms=15)
ax.scatter(init_case_list.loc[~init_case_list.in_new_region, 'center_lon'],
           init_case_list.loc[~init_case_list.in_new_region, 'center_lat'], color='red', edgecolor='k', marker='o', ms=15)

h = [ax.plot([],[],ls=ls, c=c, ms=ms, m=m) for ls, c, ms, m in zip(['-', '', '', ''], ['gray', 'tab:blue', 'r', 'gold'], [0, 15, 15, 15], ['', '.', '.', '.'])
    ]
ax.legend(h, ['Old region', 'Kept points', 'Out of region', 'Out of time range'], loc='ur', ncols=1, alpha=1)

In [ ]:
# Number of samples that should come from each month
init_case_list['month'] = init_case_list.start_date.dt.month
init_counts = (init_case_list.groupby(['region', 'month']).count()).pivot_table(index='region', columns='month', values='center_x').fillna(0).astype(int)
init_counts

filter_counts = (init_case_list.loc[init_case_list.in_new_region & init_case_list.in_time_range].groupby(['region', 'month']).count()).pivot_table(index='region', columns='month', values='center_x').fillna(0).astype(int)
filter_counts

### Simple cases
Baffin Bay, Beaufort Sea, East Siberian Sea, Sea of Okhostk, and Laptev Sea all are only reduced in size: thus, we keep all samples from the original that are still in the time range, and draw new samples to bring the numbers back to the init counts.

Where we've only discarded samples from restricting the area and date range, we add new samples to bring the numbers to the expected count. We draw the maximum number of samples needed by month, then take the samples in the order they were drawn.

In [ ]:
# Case numbers from init list to keep
keep_init = []

# Newly generated samples
new_samples = []
for region in ['baffin_bay', 'beaufort_sea', 'east_siberian_sea', 'sea_of_okhostk', 'laptev_sea']:
    num_new = init_counts.loc[region,:] - filter_counts.loc[region,:]
    n = 10*max(num_new)
    candidates = find_candidates(n_samples=n,
                    region_info=regions.loc[region],
                    dx=100,
                    sic_path=sic_dataloc,
                    sif_threshold=0.25,
                    margin=50, start_year=2006, end_year=2022,
                    min_samples_per_month=max(num_new))
    
    new_sample_indices = []
    for month in num_new.index:
        count = 0
        cases = candidates.loc[candidates.month == month].index.values
        while count < num_new[month]:
            if len(cases) > count:
                new_sample_indices.append(cases[count])
            count += 1
    
    new_samples.append(candidates.loc[new_sample_indices].copy())

    # Keep the filtered original sample
    for cn in init_case_list.loc[(init_case_list.region == region) & (init_case_list.in_new_region & init_case_list.in_time_range)].index:
        keep_init.append(cn)


# Complex cases
For cases where is a shift in the coverage area, we draw a full new sample first. Then, we keep the original samples up to the number of samples in the original region in the candidate list. Finally, we add samples from the candidate list until we have the original number of samples per month.

In [ ]:
for region in ['barents_kara_seas', 'greenland_sea', 'bering_chukchi_seas', 'hudson_bay']:
    keep_old = []
    num_per_month = init_counts.loc[region,:]
    # - filter_counts.loc[region,:]
    n = 10*max(num_per_month)
    candidates = find_candidates(n_samples=n,
                    region_info=regions.loc[region],
                    dx=100,
                    sic_path=sic_dataloc,
                    sif_threshold=0.25,
                    margin=50, start_year=2006, end_year=2022,
                    min_samples_per_month=max(num_per_month))
    
    # Identify how many to keep from the original sample 
    candidates['in_old_region'] = False
    for name, row in candidates.iterrows():
        candidates.loc[name, 'in_old_region'] = in_box(row.center_x, row.center_y, old_regions.loc[row.region, 'left_x'],
                                                       old_regions.loc[row.region, 'right_x'], old_regions.loc[row.region, 'lower_y'],
                                                       old_regions.loc[row.region, 'upper_y'])
    
    # "in_new_region" refers to samples that are outside the old region
    candidates['in_new_region'] = ~candidates['in_old_region']    

    # Get the number of candidates needed per month
    candidates['keep_idx'] = False
    months = []
    for month, group in candidates.groupby('month'):
        months.append(month)
        candidates.loc[group.index.values[0:num_per_month[month]], 'keep_idx'] = True
    candidates = candidates.loc[candidates['keep_idx'], :].copy()
    candidates.drop('keep_idx', axis=1, inplace=True)
    
    # Count number of samples in old area
    num_old_to_keep = candidates.loc[candidates.in_old_region].groupby('month').count()['region']
    num_new_to_keep = candidates.loc[~candidates.in_old_region].groupby('month').count()['region']

    # Make sure all months are in the index, even if no data in month
    num_old_to_keep = num_old_to_keep.reindex_like(pd.Series(np.nan, index=months)).fillna(0).astype(int)
    num_new_to_keep = num_new_to_keep.reindex_like(pd.Series(np.nan, index=months)).fillna(0).astype(int)
    
    keep_new = []
    for month in months:
        old_cases = init_case_list.loc[(init_case_list.in_new_region & init_case_list.in_time_range) & (init_case_list.month == month)]
        old_cases = old_cases.loc[old_cases.region == region].index
        # If we need more samples in the old region than we have left,
        # keep all the original samples
        if num_old_to_keep[month] >= filter_counts.loc[region, month]:
            if filter_counts.loc[region, month] > 0:
                for cn in old_cases:
                    keep_old.append(cn)
        # if we have too many samples from the old region, keep the number we need
        elif num_old_to_keep[month] < filter_counts.loc[region, month]:
            for cn in old_cases[0:num_old_to_keep[month]]:
                keep_old.append(cn)
        # Add new samples from the old region to bring the number up to the right amount
        if num_old_to_keep[month] - filter_counts.loc[region, month] > 0:
            n_new = num_old_to_keep[month] - filter_counts.loc[region, month]
            for idx in candidates.loc[candidates.in_old_region & (candidates.month == month)].index[0:n_new]:
                keep_new.append(idx)

        # Finally, add samples from the new region
        # The total from the new and old samples across all months should be the same as the sum of the init counts!
        # Doesn't add up for B-K seas -- recheck logic!
        if month in num_new_to_keep:
            n_new = num_new_to_keep[month]
            if n_new > 0:
                for idx in candidates.loc[~candidates.in_old_region & (candidates.month == month)].index[0:n_new]:
                    keep_new.append(idx)

    results = pd.DataFrame({'OldRegion': num_old_to_keep, 'NewRegion': num_new_to_keep,
              'KeptOld': init_case_list.loc[keep_old].groupby('month').count()['region'],
              'KeptNew': candidates.loc[keep_new].groupby('month').count()['region']}).fillna(0).astype(int)
    results['InitSampleSize'] = init_counts.loc[region]
    results['FinalSampleSize'] = results['KeptOld'] + results['KeptNew']
    if ~np.all(results['InitSampleSize'] == results['FinalSampleSize']):
        print(region, 'sample incomplete')
        
    new_samples.append(candidates.loc[keep_new,:].copy())
    keep_init += keep_old

# Merge into a dataframe
columns = ['region', 'start_date', 'center_lon', 'center_lat', 'center_x',
       'center_y', 'month', 'sea_ice_fraction', 'mean_sea_ice_concentration', 'init_case_number']

new_samples = pd.concat(new_samples)
new_samples['init_case_number'] = -1
init_case_list['init_case_number'] = init_case_list.index
new_case_list = pd.concat([init_case_list.loc[keep_init, columns], new_samples.loc[:, columns]], axis=0)
new_case_list = new_case_list.sort_values(['region', 'center_lon']).reset_index(drop=True)
new_case_list['case_number'] = [str(int(x + 1)).zfill(3) for x in new_case_list.index]
new_case_list['start_date'] = [pd.to_datetime(x) for x in new_case_list['start_date']]

column_order = ['case_number', 'region', 'start_date', 'center_lon', 'center_lat', 'center_x',
       'center_y', 'month', 'sea_ice_fraction', 'mean_sea_ice_concentration',
       'init_case_number']
new_case_list.loc[:, column_order].to_csv('../data/metadata/validation_dataset_case_list.csv')

# Plot new samples as white dots, retained old samples as solid colors
pplt.rc['reso'] = 'med'
pplt.rc['cartopy.circular'] = False
colors = {region: c['color'] for region, c in zip(
            regions.index,
            pplt.Cycle('colorblind10', len(regions)))}


crs = ccrs.CRS('epsg:3413')
fig, ax = pplt.subplots(width=4.5, proj='npstere', proj_kw={'lon_0': -45})
ax.format(land=True, color='k', boundinglat=52, landzorder=0, latmax=90)
ax.set_extent([-3.5e6, 2.8e6, -4e6, 4.4e6], crs=crs)

for region in regions.index:
  
    xbox = np.array([regions.loc[region, coord] for coord in ['left_x', 'left_x', 'right_x', 'right_x', 'left_x']])
    ybox = np.array([regions.loc[region, coord] for coord in ['lower_y', 'upper_y', 'upper_y', 'lower_y', 'lower_y']])
    ax.plot(xbox, ybox, transform=crs, color=colors[region])
    ax.scatter(new_case_list.loc[new_case_list.region == region, 'center_lon'],
                new_case_list.loc[new_case_list.region == region, 'center_lat'], color=colors[region])
    ax.scatter(new_case_list.loc[(new_case_list.region == region) & (new_case_list.init_case_number < 0), 'center_lon'],
                new_case_list.loc[(new_case_list.region == region) & (new_case_list.init_case_number < 0), 'center_lat'], edgecolor=colors[region], facecolor='w')

# Drawing new samples based on the MASIE results

I first checked whether any of the original sample were available as "back up" choices. Unfortunately, all of these were before 2006, so the MASIE data isn't available. Next step is to use the sampling routine.

 Samples to replace:
| Number | Regions | Date | 
| --- | --- | --- |
| 38 |	barents_kara_seas |	2016-08-29	|
| 40 |	barents_kara_seas |	2011-09-25	|
| 102 |	east_siberian_sea |	2011-07-13	|
| 135 |	hudson_bay		 |	2006-07-18	|
| 175 |	sea_of_okhostk |	2006-06-29	|
| 64 | bering_chukchi_seas | 2007-07-27 |
| 83 | bering_chukchi_seas | 2011-07-15 |
| 106 | greenland_sea | 2017-09-13 |
| 134 | hudson_bay | 2010-08-12 |
| 136 | hudson_bay | 2006-08-02 |

To find the new samples, I ran the sampling routine, then examined the Worldview imagery for that date to make sure the summer SIC bias wasn't affecting the choice.


In [ ]:
init_case_list['new_case_number'] = np.nan
for row, data in updated_list.iterrows():
    if data.init_case_number > 0:
        init_case_list.loc[data.init_case_number, 'new_case_number'] = row

# Checking if any of the original sample can be used.
# In both cases, my finding was no, none of the original sample was appropriate (no ice)
init_case_list.loc[(init_case_list.new_case_number.isnull() & (init_case_list.month == 7)) &\
                    (init_case_list.start_date.dt.year >= 2006)]

In [ ]:
candidates = find_candidates(n_samples=n,
                region_info=regions.loc[region],
                dx=100,
                sic_path=sic_dataloc,
                sif_threshold=0.25,
                margin=50, start_year=2006, end_year=2022,
                min_samples_per_month=n)

In [ ]:
# Merging to make the IFTP case list
case_list = pd.read_csv('../data/metadata/iftp_case_list.csv', index_col=0)

new_samples = pd.DataFrame(data=[ 
    [38,'barents_kara_seas','2014-08-02',88.339718,78.769967,887500.0,837500.0,1.0,0.752],
    [40,'barents_kara_seas','2007-09-12',12.757532,81.149074,812500.0,-512500.0,0.6,0.696],
    [102,'east_siberian_sea','2022-07-02',155.013567,71.609279,-687500.0,1887500.0,1.0,0.855],
    [135,'hudson_bay','2017-07-06',-91.780642,59.221811,-2487500.0,-2337500.0,0.64,0.43],
    [175,'sea_of_okhostk','2010-06-04',139.6859,55.708056,-312500.0,3812500.0,0.8,0.536],
    [64,'bering_chukchi_seas','2015-07-16',-167.047235,73.368234,-1537500.0,962500.0,0.56,0.455],
    [83,'bering_chukchi_seas','2018-07-26',-161.737110,72.937768,-1662500.0,837500.0,0.96,0.497],
    [106,'greenland_sea','2014-09-11',-13.431029,78.793415,637500.0,-1037500.0,0.92,0.715],
    [134,'hudson_bay','2015-08-10',-85.749345,66.119475,-1712500.0,-1987500.0,1.0,0.732],
    [136,'hudson_bay','2012-08-14',-84.364059,65.442785,-1712500.0,-2087500.0,1.0,0.806]],
                        columns=['case_number', 	'location', 	'startdate', 	'center_lon',
                                 'center_lat', 	'center_x',  'center_y', 	'sea_ice_fraction',
                                 'mean_sea_ice_concentration'])
new_samples['enddate'] = [(pd.to_datetime(sd) + pd.to_timedelta('1d')).strftime('%Y-%m-%d') for sd in new_samples['startdate']]
new_samples['left_x'] = new_samples['center_x'] - 50e3 
new_samples['right_x'] = new_samples['center_x'] + 50e3 
new_samples['lower_y'] = new_samples['center_y'] - 50e3
new_samples['top_y'] = new_samples['center_y'] + 50e3
new_samples['dx'] = '100km'
column_order = ['fullname', 'case_number', 'location', 'dx', 'left_x', 'right_x',
       'lower_y', 'top_y', 'center_x', 'center_y', 'center_lat', 'center_lon',
       'startdate', 'enddate']
new_samples['fullname'] = ['-'.join([str(cn).zfill(3), rg, '100km', sd]) for cn, rg, sd in zip(new_samples.case_number,
                                                                                             new_samples.location,
                                                                                             new_samples.startdate)]
keep_idx = [x for x in case_list.index if case_list.loc[x, 'case_number'] not in new_samples['case_number'].values]
pd.concat([case_list.loc[keep_idx], new_samples]).sort_values('case_number').to_csv('../data/metadata/iftp_case_list_updated.csv', index=False)